# Load in Data

In [ ]:
import pandas as pd

# Load features
features_df = pd.read_csv('../output/features.csv', index_col=0)

# Load selected feature names
with open('../output/selected_features_top50.txt', 'r') as f:
    selected_features = [line.strip() for line in f if line.strip()]

# features_df = features_df[selected_features + ['DQ_TARGET']]
# features_df = features_df[features_df["DQ_TARGET"].notnull()].copy()

print(f'Total consumers: {features_df.shape[0]}')
print(f'Selected features: {len(selected_features)}')
print(f'Delinquency rate: {features_df["DQ_TARGET"].mean():.2%}')

# Scoring Exclusion

In [ ]:
import numpy as np
import pandas as pd

df_work = features_df.copy()
LABEL = "DQ_TARGET"

start_total   = len(df_work)
start_labeled = df_work[LABEL].notnull().sum()

# Accumulate boolean exclusion masks — all evaluated on the ORIGINAL df_work
exclusion_masks = {}

print("Starting total  :", start_total)
print("Starting labeled:", start_labeled)

### Minimum History Days

In [ ]:
MIN_DAYS = 30

exclusion_masks["rule_min_days"] = (
    (df_work["account__history_days__all"] < MIN_DAYS).fillna(True)
)

print(f"Rule 1 (account__history_days__all < {MIN_DAYS}): "
      f"{exclusion_masks['rule_min_days'].sum()} would be removed, about {exclusion_masks['rule_min_days'].sum()/features_df.shape[0]}% of the total population")

### Minimum Total transactions

In [ ]:
MIN_TX = 10

exclusion_masks["rule_min_days"] = (
    (df_work["tx__n__all"] < MIN_TX).fillna(True)
)

print(f"Rule 2 (tx__n__all < {MIN_TX}): "
      f"{exclusion_masks['rule_min_days'].sum()} would be removed, about {exclusion_masks['rule_min_days'].sum()/features_df.shape[0]}% of the total population")

### Minimum Recent Transactions

In [ ]:
MIN_TX_30D = 5

exclusion_masks["rule_min_tx30d"] = (
    (df_work["n_tx__30d"] < MIN_TX_30D).fillna(True)
)

print(f"Rule 2 (n_tx__30d < {MIN_TX_30D}): "
      f"{exclusion_masks['rule_min_tx30d'].sum()} would be removed, about {exclusion_masks['rule_min_tx30d'].sum()/features_df.shape[0]}% of the total population")

### Income Signal Present

In [ ]:
MIN_INCOME_FREQ = 2/90  # 2 every 90 days

exclusion_masks["rule_income_presence"] = (
    (df_work["income__frequency__90d"] < MIN_INCOME_FREQ).fillna(True)
)

print(f"Rule 3 (income__frequency__90d < {MIN_INCOME_FREQ:.4f}): "
      f"{exclusion_masks['rule_income_presence'].sum()} would be removed")

### Credit transaction info present

In [ ]:
# exclusion_masks["rule_has_credit"] = (
#     (df_work["tx__max_credit__all"] <= 0).fillna(True)
# )

# print(f"Rule 4 (tx__max_credit__all <= 0): "
#       f"{exclusion_masks['rule_has_credit'].sum()} would be removed")

In [ ]:
# Combine all masks — a consumer is excluded if ANY rule flags them
combined_mask = pd.concat(exclusion_masks.values(), axis=1).any(axis=1)

# Apply once at the end
df_work = df_work.loc[~combined_mask].copy()

final_total   = len(df_work)
final_labeled = df_work[LABEL].notnull().sum()
total_removed = int(combined_mask.sum())

print("----- SUMMARY -----")
print(f"Started with : {start_total}")
print(f"Total removed: {total_removed}  ({total_removed/start_total:.1%} of original)")
print(f"Remaining    : {final_total}")
print()
print("Flagged by each rule (consumers may overlap):")
for rule, mask in exclusion_masks.items():
    print(f"  {rule}: {int(mask.sum())}")
print()
print(f"Labeled remaining: {final_labeled}")
print(f"Delinquency rate  : {df_work[LABEL].mean():.2%}")

In [ ]:
# save the final filtered dataset for modeling
df_work.to_csv('../output/filtered_features.csv', index=False)

# Data Splitting

In [ ]:
from sklearn.model_selection import train_test_split
import numpy as np

LABEL = "DQ_TARGET"

df_clean = df_work[df_work[LABEL].notna()].copy()
X = df_clean.drop(columns=[LABEL]).fillna(0).values
y = df_clean[LABEL].astype(int).values

# 60% train, 40% temp
X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)

# 20% val, 20% test
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

print("Train pos rate:", y_train.mean())
print("Val pos rate:", y_val.mean())
print("Test pos rate:", y_test.mean())

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:", np.bincount(y_train))
print("After SMOTE: ", np.bincount(y_train_sm))


In [ ]:
# import pandas as pd
# from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

# def metrics_df(model, X, y, split_name, model_name, train_time_sec):
#     y_pred = model.predict(X)

#     # ROC AUC requires probabilities
#     y_prob = model.predict_proba(X)[:, 1]
#     roc = roc_auc_score(y, y_prob)

#     return pd.DataFrame({
#         "model": [model_name],
#         "split": [split_name],
#         "accuracy": [accuracy_score(y, y_pred)],
#         "roc_auc": [roc],
#         "f1": [f1_score(y, y_pred, zero_division=0)],
#         "train_time_sec": [train_time_sec],
#     })

# Model Performance

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
import numpy as np

_neg, _pos = (y_train == 0).sum(), (y_train == 1).sum()
_sw = np.where(y_train == 1, _neg / _pos, 1.0)

# Scaler for class-weight models (fit on original train)
_sc1 = StandardScaler()
_Xtr1 = _sc1.fit_transform(X_train)
_Xva1 = _sc1.transform(X_val)
_Xte1 = _sc1.transform(X_test)

# Scaler for SMOTE models (fit on SMOTE-resampled train)
_sc2 = StandardScaler()
_sc2.fit(X_train_sm)
_Xtr2     = _sc2.transform(X_train)    # original train — for fair eval
_Xva2     = _sc2.transform(X_val)
_Xte2     = _sc2.transform(X_test)
_Xtr2_sm  = _sc2.transform(X_train_sm) # SMOTE train — for fitting

# ── Logistic Regression ──────────────────────────────────────────
_lr_cw = LogisticRegression(C=0.1, solver='saga', l1_ratio=1.0, penalty='elasticnet',
                             max_iter=3000, class_weight='balanced', random_state=42)
_lr_cw.fit(_Xtr1, y_train)

_lr_sm = LogisticRegression(C=0.1, solver='saga', l1_ratio=1.0, penalty='elasticnet',
                             max_iter=3000, random_state=42)
_lr_sm.fit(_Xtr2_sm, y_train_sm)

# ── Gradient Boosting ────────────────────────────────────────────
_gbm_cw = GradientBoostingClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                                      subsample=0.8, min_samples_leaf=30, random_state=42)
_gbm_cw.fit(X_train, y_train, sample_weight=_sw)

_gbm_sm = GradientBoostingClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
                                      subsample=0.8, min_samples_leaf=30, random_state=42)
_gbm_sm.fit(X_train_sm, y_train_sm)

# ── Results (all evaluated on ORIGINAL, non-oversampled data) ────
print(f"{'Model':<32} {'Train AUC':>10} {'Val AUC':>10} {'Test AUC':>10}")
print("-" * 64)
for _name, _mdl, _Xtr_, _Xva_, _Xte_ in [
    ("LogReg  (class weight)",  _lr_cw,  _Xtr1,  _Xva1, _Xte1),
    ("LogReg  (SMOTE)",         _lr_sm,  _Xtr2,  _Xva2, _Xte2),
    ("GBM     (sample weight)", _gbm_cw, X_train, X_val, X_test),
    ("GBM     (SMOTE)",         _gbm_sm, X_train, X_val, X_test),
]:
    a = [roc_auc_score(y, _mdl.predict_proba(X)[:, 1])
         for y, X in [(y_train, _Xtr_), (y_val, _Xva_), (y_test, _Xte_)]]
    print(f"{_name:<32} {a[0]:>10.4f} {a[1]:>10.4f} {a[2]:>10.4f}")


# Decision Tree

In [ ]:
from sklearn.tree import DecisionTreeClassifier
import time

t0 = time.time()
dt = DecisionTreeClassifier(
    max_depth=6,
    min_samples_leaf=25,
    random_state=42
)
dt.fit(X_train_sm, y_train_sm)
dt_train_time = time.time() - t0

dt_results = pd.concat([
    metrics_df(dt, X_train, y_train, "train", "DecisionTree", dt_train_time),
    metrics_df(dt, X_val, y_val, "validation", "DecisionTree", dt_train_time),
    metrics_df(dt, X_test, y_test, "test", "DecisionTree", dt_train_time),
], ignore_index=True)

dt_results

# Random Forest

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import time

t0 = time.time()
rf = RandomForestClassifier(
    n_estimators=400,
    min_samples_leaf=10,
    n_jobs=-1,
    random_state=42
)
rf.fit(X_train_sm, y_train_sm)
rf_train_time = time.time() - t0

rf_results = pd.concat([
    metrics_df(rf, X_train, y_train, "train", "RandomForest", rf_train_time),
    metrics_df(rf, X_val, y_val, "validation", "RandomForest", rf_train_time),
    metrics_df(rf, X_test, y_test, "test", "RandomForest", rf_train_time),
], ignore_index=True)

rf_results

In [ ]:
all_results = pd.concat([dt_results, rf_results], ignore_index=True)
all_results = all_results.drop(columns=["train_time_sec"], errors="ignore")
all_results

In [ ]:
import pandas as pd
import numpy as np

# Decision Tree baseline (fill these in)
dt_baseline = pd.DataFrame({
    "model": ["DecisionTree", "DecisionTree", "DecisionTree"],
    "split": ["train", "validation", "test"],
    "accuracy": [0.752, 0.721, 0.711],      
    "roc_auc": [0.69, 0.621, 0.629],       
    "f1": [0.24, 0.165, 0.164],            
    "train_time_sec": [2.51, 2.51, 2.51] 
})

# Random Forest baseline (your values)
rf_baseline = pd.DataFrame({
    "model": ["RandomForest", "RandomForest", "RandomForest"],
    "split": ["train", "validation", "test"],
    "accuracy": [0.96, 0.87, 0.88],
    "roc_auc": [0.9850, 0.7925, 0.7663],
    "f1": [0.79, 0.29, 0.29],
    "train_time_sec": [2.86, 2.86, 2.86]
})

baseline_results = pd.concat([dt_baseline, rf_baseline], ignore_index=True)
baseline_results = baseline_results.drop(columns=["train_time_sec"], errors="ignore")
baseline_results